# VisionAttend AI: Deep Learning Face Recognition Training Pipeline
### Enterprise Biometric Attendance & Identity Verification System

This Google Colab notebook provides the complete end-to-end Deep Learning training pipeline for the **AI Attendance Manager**. 

#### Objectives & Requirements:
1. **Dataset Ingestion**: Unpack and validate student face datasets captured via webcam (100+ images per student).
2. **Data Augmentation**: Apply real-time classroom lighting and angle transforms (rotation, zoom, brightness, horizontal flip).
3. **Deep Learning Architecture**: Implement **MobileNetV2** Transfer Learning for fast, robust feature extraction (< 2.0s face inference latency).
4. **Model Training & Fine-Tuning**: Optimize classification head using Adam optimizer with Early Stopping and Learning Rate Decay.
5. **Comprehensive Evaluation**: Generate Training/Validation Curves, Confusion Matrix, and Classification Reports.
6. **Artifact Export**: Export `attendance_model.h5` and `labels.json` directly into the web application's `models/` directory.


## Step 1: Environment Setup & Hardware Accelerator Check
Verify TensorFlow version and check for active GPU acceleration (NVIDIA T4 / V100 on Google Colab).


In [ ]:
import os
import zipfile
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print(f"[*] TensorFlow Version: {tf.__version__}")

# Check GPU availability
gpu_devices = tf.config.list_physical_devices("GPU")
if gpu_devices:
    print(f"[SUCCESS] GPU Accelerator Detected: {gpu_devices[0]}")
    print("[*] Ready for high-speed Deep Learning training.")
else:
    print("[INFO] No dedicated GPU detected. Training will run on standard CPU.")
    print("[TIP] On Google Colab, go to 'Runtime' -> 'Change runtime type' -> Select 'T4 GPU'.")


## Step 2: Dataset Ingestion & Class Verification
Unpack `dataset.zip` containing registered student directories (e.g., `101_Ali`, `102_Sara`) and verify sample counts.


In [ ]:
DATASET_ZIP = "dataset.zip"
DATASET_DIR = "dataset"

# If running on Colab and dataset.zip exists, unzip it
if os.path.exists(DATASET_ZIP):
    print(f"[*] Extracting '{DATASET_ZIP}'...")
    with zipfile.ZipFile(DATASET_ZIP, "r") as zip_ref:
        zip_ref.extractall(".")
    print("[SUCCESS] Dataset successfully unpacked.")
elif os.path.exists(DATASET_DIR):
    print(f"[*] Using existing '{DATASET_DIR}' directory.")
else:
    print("[!] Note: If running on Colab, please upload 'dataset.zip' using the Files panel on the left.")

# Verify enrolled classes and image samples
if os.path.exists(DATASET_DIR):
    classes = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
    print("\n" + "="*50)
    print(f"   ENROLLED STUDENTS DATASET SUMMARY ({len(classes)} Classes)")
    print("="*50)
    total_images = 0
    for idx, c in enumerate(classes, 1):
        class_folder = os.path.join(DATASET_DIR, c)
        imgs = [f for f in os.listdir(class_folder) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        total_images += len(imgs)
        status = "[READY]" if len(imgs) >= 50 else "[LOW SAMPLES]"
        print(f" {idx:2d}. {c:<25} -> {len(imgs):4d} images {status}")
    print("-" * 50)
    print(f" Total Dataset Samples: {total_images} images")
    print("="*50)
else:
    print("[!] Dataset directory not yet available.")


## Step 3: Classroom Data Augmentation & Preprocessing
Data augmentation expands the model's generalization capacity by introducing lighting variations, subtle face tilts, and scale shifts matching real-world classroom conditions.


In [ ]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 32

# Robust augmentation simulating diverse ambient illumination and student head poses
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    validation_split=0.20  # 80% Training, 20% Validation split
)

if os.path.exists(DATASET_DIR) and len(os.listdir(DATASET_DIR)) > 0:
    train_generator = train_datagen.flow_from_directory(
        DATASET_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        subset="training",
        shuffle=True
    )

    val_generator = train_datagen.flow_from_directory(
        DATASET_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        subset="validation",
        shuffle=False
    )

    num_classes = train_generator.num_classes
    print(f"\n[SUCCESS] Data pipeline ready: {train_generator.samples} training images, {val_generator.samples} validation images across {num_classes} classes.")
else:
    print("[!] Please ensure dataset/ has student images to build generators.")


## Step 4: Export Class Mapping Dictionary (`labels.json`)
The label dictionary maps softmax prediction output indices to human-readable Student IDs and Names for the Flask application.


In [ ]:
os.makedirs("models", exist_ok=True)

if 'train_generator' in locals():
    class_indices = train_generator.class_indices
    index_to_label = {v: k for k, v in class_indices.items()}

    with open("models/labels.json", "w") as f:
        json.dump(index_to_label, f, indent=4)

    print("[SUCCESS] 'models/labels.json' created successfully:")
    print(json.dumps(index_to_label, indent=2))
else:
    print("[!] Generator not loaded; skipping label export.")


## Step 5: MobileNetV2 Deep Neural Network Architecture
We employ **MobileNetV2** pre-trained on ImageNet. Its inverted residual structure and linear bottlenecks provide cutting-edge accuracy with minimal computational overhead, satisfying the SRS requirement of **< 2.0s face detection latency**.


In [ ]:
def build_attendance_model(num_classes):
    # Load base MobileNetV2 without top fully connected layers
    base_model = MobileNetV2(
        input_shape=(160, 160, 3),
        include_top=False,
        weights="imagenet"
    )
    
    # Freeze base model weights during initial training phase
    base_model.trainable = False

    # Build custom classification head for student biometric recognition
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.4)(x)
    x = BatchNormalization()(x)
    predictions = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs=base_model.input, outputs=predictions, name="VisionAttend_Face_Classifier")
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model, base_model

if 'num_classes' in locals():
    model, base_model = build_attendance_model(num_classes)
    model.summary()
else:
    print("[!] num_classes not defined. Run dataset generator step first.")


## Step 6: Model Training with Automated Callbacks
We employ automated callbacks:
- **`ModelCheckpoint`**: Persists the highest-performing validation accuracy weights to `models/attendance_model.h5`.
- **`EarlyStopping`**: Halts training if validation loss plateaus to prevent overfitting.
- **`ReduceLROnPlateau`**: Automatically decreases learning rate when loss stabilizes to hone convergence.


In [ ]:
EPOCHS = 20

callbacks = [
    ModelCheckpoint(
        filepath="models/attendance_model.h5",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

if 'model' in locals() and 'train_generator' in locals():
    print("[*] Initiating model training...")
    history = model.fit(
        train_generator,
        epochs=EPOCHS,
        validation_data=val_generator,
        callbacks=callbacks
    )
    print("[SUCCESS] Phase 1 Training completed successfully.")
else:
    print("[!] Model or generator not initialized.")


## Step 7: Training & Validation Performance Visualization
Generate high-resolution training curves illustrating Accuracy and Categorical Crossentropy Loss across all training epochs.


In [ ]:
if 'history' in locals():
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Accuracy Plot
    ax1.plot(history.history['accuracy'], label='Train Accuracy', color='#10B981', linewidth=2.5)
    ax1.plot(history.history['val_accuracy'], label='Val Accuracy', color='#3B82F6', linewidth=2.5, linestyle='--')
    ax1.set_title('Model Accuracy vs Epochs', fontsize=13, fontweight='bold', pad=12)
    ax1.set_xlabel('Epoch', fontsize=11)
    ax1.set_ylabel('Accuracy', fontsize=11)
    ax1.legend(loc='lower right', frameon=True)
    ax1.grid(True, alpha=0.3)

    # Loss Plot
    ax2.plot(history.history['loss'], label='Train Loss', color='#EF4444', linewidth=2.5)
    ax2.plot(history.history['val_loss'], label='Val Loss', color='#F59E0B', linewidth=2.5, linestyle='--')
    ax2.set_title('Model Loss vs Epochs', fontsize=13, fontweight='bold', pad=12)
    ax2.set_xlabel('Epoch', fontsize=11)
    ax2.set_ylabel('Loss', fontsize=11)
    ax2.legend(loc='upper right', frameon=True)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("models/training_curves.png", dpi=300)
    plt.show()
    print("[SUCCESS] Performance curves rendered and saved to 'models/training_curves.png'.")
else:
    print("[!] No training history available to plot.")


## Step 8: Confusion Matrix & Metric Evaluation
Compute the Confusion Matrix Heatmap, Precision, Recall, and F1-Score for each enrolled student class to validate recognition reliability.


In [ ]:
if 'model' in locals() and 'val_generator' in locals():
    val_generator.reset()
    predictions = model.predict(val_generator, verbose=1)
    y_pred = np.argmax(predictions, axis=1)
    y_true = val_generator.classes
    class_labels = list(val_generator.class_indices.keys())

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=class_labels, yticklabels=class_labels)
    plt.title("Confusion Matrix - Student Biometric Recognition", fontsize=13, fontweight='bold', pad=12)
    plt.xlabel("Predicted Student", fontsize=11)
    plt.ylabel("Actual Student", fontsize=11)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig("models/confusion_matrix.png", dpi=300)
    plt.show()

    # Classification Report
    print("\n" + "="*60)
    print("               DETAILED CLASSIFICATION REPORT")
    print("="*60)
    print(classification_report(y_true, y_pred, target_names=class_labels, zero_division=0))
    print("="*60)
else:
    print("[!] Model or validation generator not available for evaluation.")


## Step 9: TFLite Optimization & Artifact Packaging
Convert the trained model to lightweight TensorFlow Lite format for real-time edge inference (< 50ms per frame), and bundle all deliverables into `models_export.zip`.


In [ ]:
import shutil

# 1. Convert to TFLite for edge deployment
try:
    print("[*] Converting MobileNetV2 to TFLite format for ultra-fast edge inference...")
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    with open("models/attendance_model.tflite", "wb") as f:
        f.write(tflite_model)
    size_mb = len(tflite_model) / (1024 * 1024)
    print(f"[SUCCESS] Exported 'models/attendance_model.tflite' ({size_mb:.2f} MB)")
except Exception as e:
    print(f"[!] TFLite conversion note: {e}")

# 2. Package all model artifacts and evaluation graphs
shutil.make_archive("models_export", "zip", "models")
print("[SUCCESS] Created all-in-one package 'models_export.zip' containing:")
print("    - attendance_model.h5")
print("    - attendance_model.tflite")
print("    - labels.json")
print("    - training_curves.png")
print("    - confusion_matrix.png")


## Step 10: Download Model Deliverables
Download `models_export.zip` containing all trained weights, label mappings, and evaluation metrics.


In [ ]:
try:
    from google.colab import files
    print("[*] Initiating automatic download from Google Colab...")
    if os.path.exists("models_export.zip"):
        files.download("models_export.zip")
        print("[SUCCESS] Downloading 'models_export.zip'...")
    else:
        if os.path.exists("models/attendance_model.h5"):
            files.download("models/attendance_model.h5")
        if os.path.exists("models/labels.json"):
            files.download("models/labels.json")
except ImportError:
    print("[INFO] Running locally outside Google Colab.")
    print("[*] Model artifacts are already saved in the local 'models/' directory.")
